## **Aim**
To implement a program that detects unusual outbound network activity from simulated network traffic records.

## **Algorithm**
**Step 1:** Import `json`, `collections.Counter`, `datetime`, and `statistics` libraries.

**Step 2:** Create simulated network traffic logs with fields: timestamp, source_ip, dest_ip, source_port, dest_port, protocol, bytes_sent, bytes_received, direction (inbound/outbound).

**Step 3:** Filter for outbound connections only.

**Step 4:** Establish baselines for normal outbound traffic per host (average bytes, connection frequency, common ports).

**Step 5:** Detect anomalies:
   - Unusually large data transfers
   - Connections to rare/uncommon destination ports
   - Connections at unusual hours
   - High frequency of connections to single destination
   - Low bytes received but high bytes sent (exfiltration)
   - Connections to known malicious IP ranges

**Step 6:** Score each anomaly and generate report.

In [1]:
import json
import shutil
from collections import Counter, defaultdict
from datetime import datetime, timedelta
import statistics

SUSPICIOUS_DESTINATIONS = {
    "192.168.100.50": "Known C2 server",
    "203.0.113.45": "Tor exit node / suspicious",
    "198.51.100.23": "VPN/Proxy / suspicious",
    "192.0.2.100": "Known malicious host",
}

COMMON_PORTS = {53, 80, 443, 22, 25, 123, 389, 636}

def create_sample_traffic_log(log_file):
    now = datetime.now()
    base = now - timedelta(hours=4)
    
    events = []
    
    # Normal web browsing from 192.168.1.50
    for i in range(50):
        events.append({
            "timestamp": (base + timedelta(minutes=i*4)).isoformat(),
            "source_ip": "192.168.1.50",
            "dest_ip": "8.8.8.8" if i % 3 == 0 else "1.1.1.1",
            "source_port": 1024 + i,
            "dest_port": 53,
            "protocol": "UDP",
            "bytes_sent": 100,
            "bytes_received": 200,
            "direction": "outbound"
        })
    
    for i in range(30):
        events.append({
            "timestamp": (base + timedelta(minutes=i*8)).isoformat(),
            "source_ip": "192.168.1.50",
            "dest_ip": f"203.0.113.{i%10}",
            "source_port": 20000 + i,
            "dest_port": 443 if i % 2 == 0 else 80,
            "protocol": "TCP",
            "bytes_sent": 500,
            "bytes_received": 5000,
            "direction": "outbound"
        })
    
    # Normal admin traffic from 192.168.1.200
    for i in range(40):
        events.append({
            "timestamp": (base + timedelta(minutes=i*6)).isoformat(),
            "source_ip": "192.168.1.200",
            "dest_ip": "192.168.1.10",
            "source_port": 30000 + i,
            "dest_port": 443 if i % 2 == 0 else 80,
            "protocol": "TCP",
            "bytes_sent": 1000,
            "bytes_received": 3000,
            "direction": "outbound"
        })
    
    # Normal SSH from 192.168.1.10
    for i in range(30):
        events.append({
            "timestamp": (base + timedelta(minutes=i*8)).isoformat(),
            "source_ip": "192.168.1.10",
            "dest_ip": "192.168.1.50",
            "source_port": 40000 + i,
            "dest_port": 22,
            "protocol": "TCP",
            "bytes_sent": 1500,
            "bytes_received": 2000,
            "direction": "outbound"
        })
    
    # Anomaly 1: Data exfiltration from 192.168.1.10 to C2
    for i in range(5):
        events.append({
            "timestamp": (base + timedelta(hours=2, minutes=i*10)).isoformat(),
            "source_ip": "192.168.1.10",
            "dest_ip": "192.168.100.50",  # Known C2
            "source_port": 50000 + i,
            "dest_port": 443,
            "protocol": "TCP",
            "bytes_sent": 10000000,  # 10MB each
            "bytes_received": 200,
            "direction": "outbound"
        })
    
    # Anomaly 2: Large transfer at 3 AM from 192.168.1.150
    for i in range(3):
        events.append({
            "timestamp": (base.replace(hour=3, minute=15) + timedelta(minutes=i*5)).isoformat(),
            "source_ip": "192.168.1.150",
            "dest_ip": "203.0.113.45",  # Tor exit
            "source_port": 60000 + i,
            "dest_port": 443,
            "protocol": "TCP",
            "bytes_sent": 3333333,
            "bytes_received": 150,
            "direction": "outbound"
        })
    
    # Anomaly 3: Internal lateral movement
    for i in range(2):
        events.append({
            "timestamp": (base + timedelta(hours=3, minutes=45)).isoformat(),
            "source_ip": "10.0.0.50",
            "dest_ip": "192.168.1.10",
            "source_port": 70000 + i,
            "dest_port": 22,
            "protocol": "TCP",
            "bytes_sent": 2500000,
            "bytes_received": 1000,
            "direction": "outbound"
        })
    
    # Anomaly 4: Unusual connection from user workstation
    events.append({
        "timestamp": (base.replace(hour=2, minute=30)).isoformat(),
        "source_ip": "192.168.1.50",
        "dest_ip": "198.51.100.23",  # Suspicious IP
        "source_port": 80000,
        "dest_port": 80,
        "protocol": "TCP",
        "bytes_sent": 500000,
        "bytes_received": 100,
        "direction": "outbound"
    })
    
    # Anomaly 5: Uncommon port
    events.append({
        "timestamp": (base + timedelta(hours=3, minutes=20)).isoformat(),
        "source_ip": "192.168.1.200",
        "dest_ip": "192.0.2.100",
        "source_port": 90000,
        "dest_port": 8080,  # Uncommon
        "protocol": "TCP",
        "bytes_sent": 1000000,
        "bytes_received": 5000,
        "direction": "outbound"
    })
    
    # Additional normal traffic for baselines
    for i in range(20):
        events.append({
            "timestamp": (base + timedelta(minutes=i*12)).isoformat(),
            "source_ip": "10.0.0.50",
            "dest_ip": "192.168.1.10",
            "source_port": 40000 + i,
            "dest_port": 443,
            "protocol": "TCP",
            "bytes_sent": 1000,
            "bytes_received": 2000,
            "direction": "outbound"
        })
    
    for i in range(10):
        events.append({
            "timestamp": (base + timedelta(minutes=i*24)).isoformat(),
            "source_ip": "192.168.1.150",
            "dest_ip": "192.168.1.10",
            "source_port": 50000 + i,
            "dest_port": 443,
            "protocol": "TCP",
            "bytes_sent": 500,
            "bytes_received": 1000,
            "direction": "outbound"
        })
    
    with open(log_file, "w") as f:
        json.dump(events, f, indent=2)

def analyze_outbound_traffic(log_file):
    with open(log_file, "r") as f:
        events = json.load(f)
    
    # Filter outbound only
    outbound = [e for e in events if e["direction"] == "outbound"]
    
    # Build baselines per source host
    host_stats = defaultdict(lambda: {
        "bytes_sent": [],
        "dest_ports": Counter(),
        "dest_ips": Counter(),
        "hours": Counter(),
        "connections": 0,
        "ratios": [],
    })
    
    for e in outbound:
        src = e["source_ip"]
        stats = host_stats[src]
        stats["bytes_sent"].append(e["bytes_sent"])
        stats["dest_ports"][e["dest_port"]] += 1
        stats["dest_ips"][e["dest_ip"]] += 1
        ts = datetime.fromisoformat(e["timestamp"])
        stats["hours"][ts.hour] += 1
        stats["connections"] += 1
        if e["bytes_received"] > 0:
            stats["ratios"].append(e["bytes_sent"] / e["bytes_received"])
    
    # Calculate baselines
    baselines = {}
    for src, stats in host_stats.items():
        if stats["connections"] > 0:
            baselines[src] = {
                "avg_bytes_sent": statistics.mean(stats["bytes_sent"]) if stats["bytes_sent"] else 0,
                "conn_per_hour": stats["connections"] / 4,  # 4 hour window
                "common_ports": set(p for p, c in stats["dest_ports"].most_common(3)),
                "common_hours": set(h for h, c in stats["hours"].most_common(6)),
                "avg_ratio": statistics.mean(stats["ratios"]) if stats["ratios"] else 1,
            }
    
    # Detect anomalies
    anomalies = []
    
    for e in outbound:
        src = e["source_ip"]
        dst = e["dest_ip"]
        dport = e["dest_port"]
        ts = datetime.fromisoformat(e["timestamp"])
        sent = e["bytes_sent"]
        recv = e["bytes_received"]
        
        score = 0
        indicators = []
        
        baseline = baselines.get(src, {})
        
        # Large data transfer
        if baseline and sent > baseline.get("avg_bytes_sent", 0) * 50:
            score += 30
            indicators.append(f"Data transfer {sent/1024/1024:.1f}MB >> baseline {baseline['avg_bytes_sent']/1024:.1f}KB")
        
        # High send/receive ratio (exfiltration)
        if recv > 0 and sent / recv > 100:
            score += 25
            indicators.append(f"High send/receive ratio: {sent/recv:.0f}:1")
        elif recv == 0 and sent > 1000:
            score += 20
            indicators.append(f"No data received, {sent/1024:.0f}KB sent")
        
        # Uncommon destination port
        if dport not in COMMON_PORTS and dport not in baseline.get("common_ports", set()):
            score += 15
            indicators.append(f"Uncommon destination port: {dport}")
        
        # Unusual hour
        if ts.hour not in baseline.get("common_hours", set()) and (ts.hour < 6 or ts.hour > 22):
            score += 15
            indicators.append(f"Connection at unusual hour: {ts.hour:02d}:00")
        
        # Known suspicious destination
        if dst in SUSPICIOUS_DESTINATIONS:
            score += 30
            indicators.append(f"Known suspicious destination: {SUSPICIOUS_DESTINATIONS[dst]}")
        
        # Internal lateral movement (RFC1918 to RFC1918 on sensitive ports)
        if (src.startswith("10.") or src.startswith("172.16.") or src.startswith("192.168.")) and \
           (dst.startswith("10.") or dst.startswith("172.16.") or dst.startswith("192.168.")) and \
           dport in [22, 3389, 445, 135, 139, 1433, 3306, 5432]:
            score += 20
            indicators.append(f"Internal lateral movement to sensitive port {dport}")
        
        # Large transfer to external
        external_dst = not (dst.startswith("10.") or dst.startswith("172.16.") or dst.startswith("192.168."))
        if external_dst and sent > 1000000:  # > 1MB to external
            score += 15
            indicators.append(f"Large external transfer: {sent/1024/1024:.1f}MB")
        
        if score > 0:
            if score >= 60:
                risk = "CRITICAL"
            elif score >= 40:
                risk = "HIGH"
            elif score >= 20:
                risk = "MEDIUM"
            else:
                risk = "LOW"
            
            anomalies.append({
                "src": src, "dst": dst, "dport": dport,
                "time": ts.isoformat(), "sent": sent, "recv": recv,
                "score": score, "risk": risk, "indicators": indicators
            })
    
    return anomalies, baselines, outbound

def main():
    log_file = "network_traffic_log.json"
    create_sample_traffic_log(log_file)
    
    print("Analyzing outbound network traffic...")
    anomalies, baselines, outbound = analyze_outbound_traffic(log_file)
    
    print(f"\n{'='*60}")
    print(f"UNUSUAL OUTBOUND NETWORK ACTIVITY REPORT")
    print(f"{'='*60}")
    print(f"Total outbound connections: {len(outbound)}")
    print(f"Unique source hosts: {len(baselines)}")
    time_range = max(datetime.fromisoformat(e["timestamp"]) for e in outbound) - \
                   min(datetime.fromisoformat(e["timestamp"]) for e in outbound)
    print(f"Time range: {time_range}")
    
    print(f"\n--- HOST BASELINES ---")
    for src, bl in baselines.items():
        print(f"{src}: avg_out={bl['avg_bytes_sent']:.0f} bytes, conn_rate={bl['conn_per_hour']:.1f}/hr, common_ports={bl['common_ports']}")
    
    # Sort anomalies by score
    anomalies.sort(key=lambda x: -x["score"])
    
    print(f"\n--- ANOMALIES DETECTED ---")
    
    for i, a in enumerate(anomalies, 1):
        print(f"\n{i}. [{a['risk']}] {a['src']} -> {a['dst']}:{a['dport']}")
        print(f"   Time: {a['time']}")
        print(f"   Bytes sent: {a['sent']:,} | Bytes received: {a['recv']:,}")
        print(f"   Ratio: {a['sent']/a['recv'] if a['recv'] else 'inf'}:1")
        print(f"   Indicators:")
        for ind in a["indicators"]:
            print(f"      - {ind}")
    
    # Summary
    from collections import Counter
    risk_counts = Counter(a["risk"] for a in anomalies)
    print(f"\n--- SUMMARY ---")
    for risk in ["CRITICAL", "HIGH", "MEDIUM", "LOW"]:
        if risk in risk_counts:
            print(f"  {risk}: {risk_counts[risk]}")

if __name__ == "__main__":
    main()

Analyzing outbound network traffic...

UNUSUAL OUTBOUND NETWORK ACTIVITY REPORT
Total outbound connections: 192
Unique source hosts: 5
Time range: 7:12:00

--- HOST BASELINES ---
192.168.1.50: avg_out=6420 bytes, conn_rate=20.2/hr, common_ports={80, 443, 53}
192.168.1.200: avg_out=25366 bytes, conn_rate=10.2/hr, common_ports={80, 443, 8080}
192.168.1.10: avg_out=1429857 bytes, conn_rate=8.8/hr, common_ports={443, 22}
192.168.1.150: avg_out=769615 bytes, conn_rate=3.2/hr, common_ports={443}
10.0.0.50: avg_out=228182 bytes, conn_rate=5.5/hr, common_ports={443, 22}

--- ANOMALIES DETECTED ---

1. [CRITICAL] 192.168.1.50 -> 198.51.100.23:80
   Time: 2026-08-20T02:30:54.993632
   Bytes sent: 500,000 | Bytes received: 100
   Ratio: 5000.0:1
   Indicators:
      - Data transfer 0.5MB >> baseline 6.3KB
      - High send/receive ratio: 5000:1
      - Known suspicious destination: VPN/Proxy / suspicious

2. [CRITICAL] 192.168.1.150 -> 203.0.113.45:443
   Time: 2026-08-20T03:15:54.993632
   Bytes

## **Result**
This the program successfully detects unusual outbound network activity from simulated network traffic records.